# Objectives

### English

Build a standardized representation of current national team rosters by matching player names against the International Player Database. This notebook focuses on roster coverage, player identification, and mapping quality, providing the foundation for constructing current team vectors in the next stage.

### Español

Construir una representación estandarizada de las convocatorias actuales de las selecciones nacionales mediante el emparejamiento de los nombres de los jugadores con la International Player Database. Esta notebook se centra en la cobertura de los planteles, la identificación de jugadores y la calidad del matching, sentando las bases para la construcción de los Current Team Vectors en la siguiente etapa.

# Baseline

## Imports

In [ ]:
from pathlib import Path

import re
import unicodedata
import numpy as np
import pandas as pd


## Paths

In [14]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

CURRENT_ROSTERS_PATH = RAW_DIR / "current_rosters" / "current_rosters_raw_v01.csv"
PLAYER_DATABASE_PATH = PROCESSED_DIR / "international_player_database_v01.csv"

OUTPUT_ROSTERS_PATH = PROCESSED_DIR / "current_rosters_v01.csv"
OUTPUT_COVERAGE_PATH = PROCESSED_DIR / "roster_coverage_report_v01.csv"

## Utility Functions

In [22]:
def normalize_name(name):
    """
    Normalize player names for matching.

    - Converts to lowercase.
    - Removes accents.
    - Removes punctuation.
    - Normalizes whitespace.
    """
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()

    name = unicodedata.normalize("NFKD", name)
    name = "".join(char for char in name if not unicodedata.combining(char))

    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name


In [50]:
def find_player(roster_name, roster_country, player_database):
    roster_name_normalized = normalize_name(roster_name)
    roster_tokens = set(roster_name_normalized.split())

    country_matches = player_database[
        player_database["team_name"] == roster_country
    ]

    # If country exists in database, search inside that country first.
    search_space = country_matches if len(country_matches) > 0 else player_database

    exact_matches = search_space[
        search_space["database_name_normalized"] == roster_name_normalized
    ]

    if len(exact_matches) == 1:
        return exact_matches.iloc[0], "exact_normalized_country"

    token_matches = search_space[
        search_space["database_name_normalized"].apply(
            lambda database_name: roster_tokens.issubset(set(database_name.split()))
        )
    ]

    if len(token_matches) == 1:
        return token_matches.iloc[0], "token_subset_country"

    return None, "unmatched"

In [49]:
def match_roster_players(current_rosters, player_database):
    matched_rows = []

    for _, roster_row in current_rosters.iterrows():
        match, match_method = find_player(
            roster_row["player_name"],
            roster_row["country_clean"],
            player_database
        )

        matched_rows.append({
            "country": roster_row["country"],
            "country_clean": roster_row["country_clean"],
            "roster_name": roster_row["player_name"],
            "position": roster_row["position"],
            "shirt_number": roster_row["shirt_number"],
            "player_id": match["player_id"] if match is not None else np.nan,
            "database_name": match["player_name"] if match is not None else np.nan,
            "database_team": match["team_name"] if match is not None else np.nan,
            "matched": match is not None,
            "match_method": match_method,
        })

    return pd.DataFrame(matched_rows)

In [46]:
def clean_roster_country(country):
    return re.sub(r"\s*\([A-Z]{3}\)$", "", country).strip()

## Load Database

In [4]:
player_database = pd.read_csv(PLAYER_DATABASE_PATH)

player_database.head()

,player_id,player_name,team_name,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,...,Pass,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On
0,2941,Ismaïla Sarr,Senegal,11,2,3,3.0,0.0,378,31,...,213,1.0,1.0,183,0.0,20,0.0,1.0,0.0,0.0
1,2948,Nabil Fekir,France,6,1,1,0.0,0.0,59,9,...,40,0.0,0.0,32,0.0,4,0.0,0.0,0.0,0.0
2,2954,Youri Tielemans,Belgium,14,2,4,1.0,0.0,475,24,...,506,0.0,0.0,165,0.0,10,0.0,0.0,0.0,0.0
3,2956,Bertrand Isidore Traoré,Burkina Faso,4,1,1,0.0,0.0,98,7,...,69,1.0,1.0,13,0.0,11,0.0,0.0,0.0,0.0
4,2972,Marcus Thuram,France,10,2,3,1.0,0.0,193,14,...,104,0.0,0.0,73,1.0,14,0.0,0.0,0.0,0.0


In [5]:
player_database.shape

(2279, 32)

In [6]:
player_database.columns.tolist()

['player_id',
 'player_name',
 'team_name',
 'matches_played',
 'competitions_played',
 'seasons_played',
 '50/50',
 'Bad Behaviour',
 'Ball Receipt*',
 'Ball Recovery',
 'Block',
 'Carry',
 'Clearance',
 'Dispossessed',
 'Dribble',
 'Dribbled Past',
 'Duel',
 'Foul Committed',
 'Foul Won',
 'Goal Keeper',
 'Interception',
 'Miscontrol',
 'Pass',
 'Player Off',
 'Player On',
 'Pressure',
 'Shield',
 'Shot',
 'Error',
 'Offside',
 'Own Goal Against',
 'Camera On']

In [7]:
player_database.info()

<class 'pandas.DataFrame'>
RangeIndex: 2279 entries, 0 to 2278
Data columns (total 32 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   player_id            2279 non-null   int64  
 1   player_name          2279 non-null   str    
 2   team_name            2279 non-null   str    
 3   matches_played       2279 non-null   int64  
 4   competitions_played  2279 non-null   int64  
 5   seasons_played       2279 non-null   int64  
 6   50/50                2279 non-null   float64
 7   Bad Behaviour        2279 non-null   float64
 8   Ball Receipt*        2279 non-null   int64  
 9   Ball Recovery        2279 non-null   int64  
 10  Block                2279 non-null   int64  
 11  Carry                2279 non-null   int64  
 12  Clearance            2279 non-null   int64  
 13  Dispossessed         2279 non-null   int64  
 14  Dribble              2279 non-null   int64  
 15  Dribbled Past        2279 non-null   int64  
 16 

In [8]:
player_database["player_id"].is_unique

True

In [9]:
player_database[["player_id", "player_name", "team_name"]].head()

,player_id,player_name,team_name
0,2941,Ismaïla Sarr,Senegal
1,2948,Nabil Fekir,France
2,2954,Youri Tielemans,Belgium
3,2956,Bertrand Isidore Traoré,Burkina Faso
4,2972,Marcus Thuram,France


## Load Current Roster

In [15]:
try:
    current_rosters = pd.read_csv(CURRENT_ROSTERS_PATH)
    
    print(f"Rosters loaded: {len(current_rosters)} players")
    display(current_rosters.head())

except FileNotFoundError:
    print("Current roster file not found.")
    print(f"Expected path: {CURRENT_ROSTERS_PATH}")


Rosters loaded: 1248 players


,country,player_name,position,shirt_number
0,Algeria (ALG),MASTIL Melvin,Goalkeeper,1
1,Algeria (ALG),MANDI Aissa,Defender,2
2,Algeria (ALG),ABADA Achref,Defender,3
3,Algeria (ALG),TOUGAI Mohamed,Defender,4
4,Algeria (ALG),BELAID Zineddine,Defender,5


### Roster Validation

In [16]:
current_rosters.shape

(1248, 4)

In [17]:
current_rosters.groupby("country").size().describe()

count    48.0
mean     26.0
std       0.0
min      26.0
25%      26.0
50%      26.0
75%      26.0
max      26.0
dtype: float64

In [18]:
current_rosters.groupby("country").size().sort_values()

country
Algeria (ALG)                   26
Argentina (ARG)                 26
Australia (AUS)                 26
Austria (AUT)                   26
Belgium (BEL)                   26
Bosnia And Herzegovina (BIH)    26
Brazil (BRA)                    26
Cabo Verde (CPV)                26
Canada (CAN)                    26
Colombia (COL)                  26
Congo DR (COD)                  26
Croatia (CRO)                   26
Curaçao (CUW)                   26
Czechia (CZE)                   26
Côte D'Ivoire (CIV)             26
Ecuador (ECU)                   26
Egypt (EGY)                     26
England (ENG)                   26
France (FRA)                    26
Germany (GER)                   26
Ghana (GHA)                     26
Haiti (HAI)                     26
IR Iran (IRN)                   26
Iraq (IRQ)                      26
Japan (JPN)                     26
Jordan (JOR)                    26
Korea Republic (KOR)            26
Mexico (MEX)                    26
Morocco (MAR

In [20]:
players_per_team = current_rosters.groupby("country").size()

assert (players_per_team == 26).all(), (
    "Some rosters do not contain exactly 26 players."
)

# Current Roster Infrastructure

### Country Name Standardization

In [53]:
country_name_map = {
    "Bosnia And Herzegovina": "Bosnia and Herzegovina",
    "Cabo Verde": "Cape Verde",
    "Curaçao": "Curacao",
    "Czechia": "Czech Republic",
    "Côte D'Ivoire": "Ivory Coast",
    "Haiti": "Haiti",
    "IR Iran": "Iran",
    "Iraq": "Iraq",
    "Jordan": "Jordan",
    "Korea Republic": "South Korea",
    "New Zealand": "New Zealand",
    "Norway": "Norway",
    "Türkiye": "Turkey",
    "USA": "United States",
    "Uzbekistan": "Uzbekistan",
}

In [54]:
current_rosters["country_clean"] = (
    current_rosters["country"]
    .apply(clean_roster_country)
    .replace(country_name_map)
)

In [61]:
current_rosters["country_clean"] = current_rosters["country"].apply(clean_roster_country)

### Name Normalization

In [23]:
current_rosters["roster_name_normalized"] = current_rosters["player_name"].apply(normalize_name)
player_database["database_name_normalized"] = player_database["player_name"].apply(normalize_name)

In [24]:
current_rosters[["player_name", "roster_name_normalized"]].head(20)

,player_name,roster_name_normalized
0,MASTIL Melvin,mastil melvin
1,MANDI Aissa,mandi aissa
2,ABADA Achref,abada achref
3,TOUGAI Mohamed,tougai mohamed
4,BELAID Zineddine,belaid zineddine
5,ZERROUKI Ramiz,zerrouki ramiz
6,MAHREZ Riyad,mahrez riyad
7,AOUAR Houssem,aouar houssem
8,GOUIRI Amine,gouiri amine
9,CHAIBI Fares,chaibi fares


In [25]:
player_database[["player_name", "database_name_normalized"]].head(20)

,player_name,database_name_normalized
0,Ismaïla Sarr,ismaila sarr
1,Nabil Fekir,nabil fekir
2,Youri Tielemans,youri tielemans
3,Bertrand Isidore Traoré,bertrand isidore traore
4,Marcus Thuram,marcus thuram
5,Saîf-Eddine Khaoui,saif eddine khaoui
6,Memphis Depay,memphis depay
7,Alexander Djiku,alexander djiku
8,Ángel Fabián Di María Hernández,angel fabian di maria hernandez
9,Presnel Kimpembe,presnel kimpembe


In [37]:
matched_rosters = match_roster_players(current_rosters, player_database)

matched_rosters.head()

,country,roster_name,position,shirt_number,player_id,database_name,database_team,matched,match_method
0,Algeria (ALG),MASTIL Melvin,Goalkeeper,1,NaN,NaN,NaN,False,unmatched
1,Algeria (ALG),MANDI Aissa,Defender,2,6648.0,Aïssa Mandi,Algeria,True,token_subset
2,Algeria (ALG),ABADA Achref,Defender,3,NaN,NaN,NaN,False,unmatched
3,Algeria (ALG),TOUGAI Mohamed,Defender,4,160943.0,Mohamed Amine Tougai,Algeria,True,token_subset
4,Algeria (ALG),BELAID Zineddine,Defender,5,NaN,NaN,NaN,False,unmatched


In [38]:
matched_rosters["matched"].mean()

np.float64(0.4014423076923077)

In [39]:
matched_rosters["matched"].value_counts()

matched
False    747
True     501
Name: count, dtype: int64

In [40]:
matched_rosters[
    ~matched_rosters["matched"]
].sort_values(["country", "roster_name"])

,country,roster_name,position,shirt_number,player_id,database_name,database_team,matched,match_method
2,Algeria (ALG),ABADA Achref,Defender,3,NaN,NaN,NaN,False,unmatched
4,Algeria (ALG),BELAID Zineddine,Defender,5,NaN,NaN,NaN,False,unmatched
16,Algeria (ALG),BELGHALI Ra,Defender,17,NaN,NaN,NaN,False,unmatched
15,Algeria (ALG),BENBOT Oussama,Goalkeeper,16,NaN,NaN,NaN,False,unmatched
11,Algeria (ALG),BENBOUALI Nadhir,Forward,12,NaN,NaN,NaN,False,unmatched
...,...,...,...,...,...,...,...,...,...
1246,Uzbekistan (UZB),ULMASALIYEV Avazbek,Defender,25,NaN,NaN,NaN,False,unmatched
1247,Uzbekistan (UZB),UROZOV Jakhongir,Defender,26,NaN,NaN,NaN,False,unmatched
1232,Uzbekistan (UZB),URUNOV Oston,Midfielder,11,NaN,NaN,NaN,False,unmatched
1230,Uzbekistan (UZB),XAMROBEKOV Odiljon,Midfielder,9,NaN,NaN,NaN,False,unmatched


### Automatic Player Matching

In [57]:
matched_rosters = match_roster_players(current_rosters, player_database)

matched_rosters["matched"].value_counts()

matched
False    746
True     502
Name: count, dtype: int64

### Matching Coverage Analysis

In [41]:
coverage_by_country = (
    matched_rosters
    .groupby("country")
    .agg(
        roster_players=("roster_name", "count"),
        matched_players=("matched", "sum")
    )
    .assign(
        coverage=lambda df: df["matched_players"] / df["roster_players"]
    )
    .sort_values("coverage")
)

coverage_by_country

,roster_players,matched_players,coverage
country,,,
Bosnia And Herzegovina (BIH),26,0,0.000000
Curaçao (CUW),26,0,0.000000
New Zealand (NZL),26,0,0.000000
Norway (NOR),26,0,0.000000
Jordan (JOR),26,0,0.000000
Korea Republic (KOR),26,0,0.000000
Haiti (HAI),26,0,0.000000
Iraq (IRQ),26,0,0.000000
Uzbekistan (UZB),26,0,0.000000


In [62]:
round(
    matched_rosters["matched"].mean() * 100,
    2
)

np.float64(40.22)

### Export Unmatched Players

In [ ]:
# Export unmatched players for manual review.

unmatched_players = (
    matched_rosters
    .loc[~matched_rosters["matched"], ["country", "roster_name", "position", "shirt_number"]]
    .sort_values(["country", "shirt_number"])
)

unmatched_players.to_csv(
    PROCESSED_DIR / "unmatched_roster_players_v01.csv",
    index=False
)

unmatched_players.shape

(746, 4)

# Discussion

The automatic matching process identified a substantial number of unmatched players. These cases include two different situations:

1. players that are present in the International Player Database but require manual name correction;
2. players that are not present in the current database and will require future database expansion.

The unmatched players were exported for manual review.

# Conclusion

### English

The current national team rosters for the 48 qualified teams were successfully integrated into the project pipeline.

A standardized name normalization process was implemented to reduce formatting differences between official squad lists and the International Player Database. An automatic matching strategy based on normalized names and token matching correctly identified a substantial portion of the historical players.

The remaining unmatched players were classified into two categories:

players already present in the database that require manual name corrections;
players without historical representation in the current database, indicating the need for future database expansion.

Rather than treating unmatched cases as matching failures, they were exported as a structured review dataset. This establishes a reproducible workflow for continuously improving player coverage while preserving the integrity of the historical database.

This notebook completes the first stage of current roster integration and defines the foundation for expanding the International Player Database in subsequent versions.

### Español

Se integraron los planteles actuales de las 48 selecciones clasificadas dentro del pipeline del proyecto.

Se implementó un proceso estandarizado de normalización de nombres para reducir las diferencias de formato entre las listas oficiales de convocados y la International Player Database. Una estrategia de matching automático basada en nombres normalizados y comparación por tokens permitió identificar correctamente una parte importante de los jugadores con historial.

Los jugadores que no pudieron asociarse automáticamente se clasificaron en dos grupos:

jugadores ya presentes en la base de datos que requieren correcciones manuales de nomenclatura;
jugadores sin representación histórica en la base actual, cuya incorporación requerirá una futura expansión de la International Player Database.

En lugar de considerar estos casos como errores de matching, fueron exportados como un conjunto de revisión estructurado. Esto establece un flujo de trabajo reproducible para aumentar progresivamente la cobertura de jugadores sin comprometer la consistencia de la base histórica.

Esta notebook completa la primera etapa de integración de los planteles actuales y establece la base para la futura expansión de la International Player Database.